# Notebook 1 — Inference (M-Series, 100 Test Samples)

Uses the **exact same 100 test questions** from Phase 1 E-series evaluation.
E-series inference already exists — this notebook runs M-series v1, v2, and raw baselines only.

**Output format:** matches Phase 1 files (`{name}_inference_100_TESTONLY.json`)
so Notebook 2 can judge E-series and M-series together.

In [4]:
# Cell 0: Config + load the 100 test questions
import os, sys, json, random, time, gc
import torch

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"  # Windows
# PROJECT_DIR = os.path.expanduser("~/kd_project")  # Linux/WSL

DATA_DIR = os.path.join(PROJECT_DIR, "data")
RUNS_DIR = os.path.join(PROJECT_DIR, "runs")

N_EVAL = 100
SEED = 42
GEN_KW = dict(max_new_tokens=2000, do_sample=False)

STUDENT_SIZES = {
    "qwen25_1p5b_base": {"path": "Qwen/Qwen2.5-1.5B", "is_instruct": False},
    "qwen25_3b_base":   {"path": "Qwen/Qwen2.5-3B",   "is_instruct": False},
}

# Load the exact 100 questions from Phase 1
QUESTIONS_FILE = os.path.join(DATA_DIR, "eval_questions_100.json")
assert os.path.exists(QUESTIONS_FILE), f"Put eval_questions_100.json in {DATA_DIR}"
with open(QUESTIONS_FILE) as f:
    q_data = json.load(f)

eval_questions = q_data["questions"]  # list of {"id": ..., "prompt": ...}
eval_ids = [q["id"] for q in eval_questions]
prompts_map = {q["id"]: q["prompt"] for q in eval_questions}

print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'NONE'}")
print(f"Loaded {len(eval_questions)} test questions (same as Phase 1)")
print(f"First 3 IDs: {eval_ids[:3]}")

GPU: NVIDIA GeForce RTX 4090
Loaded 100 test questions (same as Phase 1)
First 3 IDs: ['c609c8e7e129', '84865f14a3a3', 'e3e2fb505cf6']


In [5]:
# Cell 1: Define experiments to run inference on
# E-series inference already exists — only M-series + raw baseline here

ALL_EXPERIMENTS = {}

# ── Raw baseline (no adapter) ──
ALL_EXPERIMENTS["raw_baseline"] = {"adapter_path": None}

# ── M-series v1 (adapters in runs/) ──
for name in ["m1_additive_multi_loss", "m2_anti_curriculum", "m3_juggler",
             "m4_token_confidence_routing", "m5_section_routed",
             "m6_lora_merging", "m7_warmstart_from_e7"]:
    path = os.path.join(RUNS_DIR, name)
    if os.path.exists(path):
        ALL_EXPERIMENTS[name] = {"adapter_path": os.path.join(path, "{model}")}

# ── M1v2: 9 combos ──
for combo in ["A1","A2","A3","B1","B2","B3","C1","C2","C3"]:
    path = os.path.join(RUNS_DIR, "m1v2_additive_grid")
    if os.path.exists(path):
        ALL_EXPERIMENTS[f"m1v2_{combo}"] = {
            "adapter_path": os.path.join(path, "{model}", combo)
        }

# ── M2v2: 6 orderings ──
for ordering in ["DWE","DEW","WDE","WED","EDW","EWD"]:
    path = os.path.join(RUNS_DIR, "m2v2_sequential")
    if os.path.exists(path):
        ALL_EXPERIMENTS[f"m2v2_{ordering}"] = {
            "adapter_path": os.path.join(path, "{model}", ordering),
            "subdirs": ["final", "stage3"],
        }

# ── M3v2 Juggler ──
path = os.path.join(RUNS_DIR, "m3v2_juggler")
if os.path.exists(path):
    ALL_EXPERIMENTS["m3v2_juggler"] = {"adapter_path": os.path.join(path, "{model}")}

# ── M4 Metric-Guided ──
path = os.path.join(RUNS_DIR, "m4_metric_guided")
if os.path.exists(path):
    ALL_EXPERIMENTS["m4_metric_guided"] = {
        "adapter_path": os.path.join(path, "{model}", "stage3_ent")
    }

# ── Validate adapters ──
valid_experiments = {}
for exp_name, info in ALL_EXPERIMENTS.items():
    available = []
    for sname in STUDENT_SIZES:
        if info["adapter_path"] is None:
            available.append(sname)
            continue
        base = info["adapter_path"].format(model=sname)
        if "subdirs" in info:
            for subdir in info["subdirs"]:
                if os.path.exists(os.path.join(base, subdir, "adapter_config.json")):
                    available.append(sname)
                    break
        else:
            if os.path.exists(os.path.join(base, "adapter_config.json")):
                available.append(sname)
    if available:
        valid_experiments[exp_name] = {**info, "students": available}

print(f"Valid experiments: {len(valid_experiments)}")
for exp_name, info in valid_experiments.items():
    print(f"  {'🔵' if info['adapter_path'] is None else '✅'} {exp_name}: {', '.join(info['students'])}")

total = sum(len(v["students"]) for v in valid_experiments.values())
print(f"\nTotal inference runs: {total} ({total * N_EVAL} generations)")

Valid experiments: 25
  🔵 raw_baseline: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1_additive_multi_loss: qwen25_1p5b_base, qwen25_3b_base
  ✅ m2_anti_curriculum: qwen25_1p5b_base, qwen25_3b_base
  ✅ m3_juggler: qwen25_1p5b_base, qwen25_3b_base
  ✅ m4_token_confidence_routing: qwen25_1p5b_base, qwen25_3b_base
  ✅ m5_section_routed: qwen25_1p5b_base, qwen25_3b_base
  ✅ m6_lora_merging: qwen25_1p5b_base, qwen25_3b_base
  ✅ m7_warmstart_from_e7: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_A1: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_A2: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_A3: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_B1: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_B2: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_B3: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_C1: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_C2: qwen25_1p5b_base, qwen25_3b_base
  ✅ m1v2_C3: qwen25_1p5b_base, qwen25_3b_base
  ✅ m2v2_DWE: qwen25_1p5b_base, qwen25_3b_base
  ✅ m2v2_DEW: qwen25_1p5b_base, qwen25_3b_base
  ✅ m2v2_WDE: qwe

In [ ]:
# Cell 2: Run inference — grouped by base model, adapter swapping
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

def resolve_adapter(info, sname):
    if info["adapter_path"] is None:
        return None
    base = info["adapter_path"].format(model=sname)
    if "subdirs" in info:
        for subdir in info["subdirs"]:
            path = os.path.join(base, subdir)
            if os.path.exists(os.path.join(path, "adapter_config.json")):
                return path
    if os.path.exists(os.path.join(base, "adapter_config.json")):
        return base
    return None

@torch.inference_mode()
def run_inference_batch(model, tokenizer, questions):
    answers = []
    for q in questions:
        inputs = tokenizer(q["prompt"], return_tensors="pt", truncation=True)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        out = model.generate(**inputs, **GEN_KW)
        decoded = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = decoded[len(q["prompt"]):].lstrip() if decoded.startswith(q["prompt"]) else decoded.strip()
        answers.append(answer)
    return answers

# Check what's done — per (experiment, model_size) pair
done_pairs = set()
for exp_name in valid_experiments:
    out_json = os.path.join(DATA_DIR, f"{exp_name}_inference_{N_EVAL}_TESTONLY.json")
    if os.path.exists(out_json):
        with open(out_json) as f:
            data = json.load(f)
        for sname in data.get("models", {}):
            done_pairs.add((exp_name, sname))

remaining = {}
for exp_name, info in valid_experiments.items():
    needed = [s for s in info["students"] if (exp_name, s) not in done_pairs]
    if needed:
        remaining[exp_name] = {**info, "students": needed}

for exp_name in valid_experiments:
    if exp_name not in remaining:
        print(f"⏩ {exp_name} fully done")
    else:
        print(f"  🔄 {exp_name}: need {remaining[exp_name]['students']}")

print(f"\nRemaining: {len(remaining)} experiments")

for sname, scfg in STUDENT_SIZES.items():
    exps_for_size = [(n, i) for n, i in remaining.items() if sname in i["students"]]
    if not exps_for_size:
        continue

    print(f"\n{'='*70}")
    print(f"  Base model: {sname} ({len(exps_for_size)} experiments)")
    print(f"{'='*70}")

    tokenizer = AutoTokenizer.from_pretrained(scfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    if "7b" in sname:
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        base_model = AutoModelForCausalLM.from_pretrained(
            scfg["path"], quantization_config=bnb, device_map="auto", trust_remote_code=True)
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            scfg["path"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)

    for exp_name, info in exps_for_size:
        adapter_path = resolve_adapter(info, sname)
        out_json = os.path.join(DATA_DIR, f"{exp_name}_inference_{N_EVAL}_TESTONLY.json")

        print(f"\n  ── {exp_name} / {sname} ──")

        if adapter_path is None:
            model = base_model
            model.eval()
            print(f"    Raw model (no adapter)")
        else:
            print(f"    Adapter: {adapter_path}")
            model = PeftModel.from_pretrained(base_model, adapter_path)
            model.eval()

        t0 = time.time()
        answers = run_inference_batch(model, tokenizer, eval_questions)
        elapsed = time.time() - t0

        for j in range(0, len(answers), 25):
            print(f"    {min(j+25, len(answers))}/{len(answers)}")

        print(f"    ✅ {len(answers)} samples in {elapsed/60:.1f} min")

        # Save / update file
        if os.path.exists(out_json):
            with open(out_json) as f:
                result = json.load(f)
        else:
            result = {
                "meta": {"source": "eval_questions_100.json", "n_samples": N_EVAL, "seed": SEED},
                "models": {},
                "samples": [{"id": q["id"], "prompt": q["prompt"], "outputs": {}} for q in eval_questions],
            }

        result["models"][sname] = {"adapter": adapter_path or "raw", "path": scfg["path"]}
        for sample, answer in zip(result["samples"], answers):
            sample["outputs"][sname] = {"answer": answer}

        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"    Saved → {out_json}")

        if adapter_path is not None:
            del model
            gc.collect(); torch.cuda.empty_cache()

    del base_model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print(f"\n  ✅ {sname} complete")

print(f"\n{'='*70}")
print("✅ ALL INFERENCE COMPLETE")
print(f"{'='*70}")

⏩ raw_baseline fully done
⏩ m1_additive_multi_loss fully done
⏩ m2_anti_curriculum fully done
⏩ m3_juggler fully done
⏩ m4_token_confidence_routing fully done
  🔄 m5_section_routed: need ['qwen25_3b_base']
  🔄 m6_lora_merging: need ['qwen25_3b_base']
  🔄 m7_warmstart_from_e7: need ['qwen25_3b_base']
  🔄 m1v2_A1: need ['qwen25_3b_base']
  🔄 m1v2_A2: need ['qwen25_3b_base']
  🔄 m1v2_A3: need ['qwen25_3b_base']
  🔄 m1v2_B1: need ['qwen25_3b_base']
  🔄 m1v2_B2: need ['qwen25_3b_base']
  🔄 m1v2_B3: need ['qwen25_3b_base']
  🔄 m1v2_C1: need ['qwen25_3b_base']
  🔄 m1v2_C2: need ['qwen25_3b_base']
  🔄 m1v2_C3: need ['qwen25_3b_base']
  🔄 m2v2_DWE: need ['qwen25_3b_base']
  🔄 m2v2_DEW: need ['qwen25_3b_base']
  🔄 m2v2_WDE: need ['qwen25_3b_base']
  🔄 m2v2_WED: need ['qwen25_3b_base']
  🔄 m2v2_EDW: need ['qwen25_3b_base']
  🔄 m2v2_EWD: need ['qwen25_3b_base']
  🔄 m3v2_juggler: need ['qwen25_3b_base']
  🔄 m4_metric_guided: need ['qwen25_3b_base']

Remaining: 20 experiments

  Base model: qwen25_3

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 258.46it/s]



  ── m5_section_routed / qwen25_3b_base ──
    Adapter: C:\Users\adishalit1\Desktop\kd_project\runs\m5_section_routed\qwen25_3b_base


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

    25/100
    50/100
    75/100
    100/100
    ✅ 100 samples in 87.4 min
    Saved → C:\Users\adishalit1\Desktop\kd_project\data\m5_section_routed_inference_100_TESTONLY.json

  ── m6_lora_merging / qwen25_3b_base ──
    Adapter: C:\Users\adishalit1\Desktop\kd_project\runs\m6_lora_merging\qwen25_3b_base


...

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

    25/100
    50/100
    75/100
    100/100
    ✅ 100 samples in 15.5 min
    Saved → C:\Users\adishalit1\Desktop\kd_project\data\m3v2_juggler_inference_100_TESTONLY.json

  ── m4_metric_guided / qwen25_3b_base ──
    Adapter: C:\Users\adishalit1\Desktop\kd_project\runs\m4_metric_guided\qwen25_3b_base\stage3_ent


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

    25/100
    50/100
    75/100
    100/100
    ✅ 100 samples in 17.8 min
    Saved → C:\Users\adishalit1\Desktop\kd_project\data\m4_metric_guided_inference_100_TESTONLY.json

  ✅ qwen25_3b_base complete

✅ ALL INFERENCE COMPLETE


: 